# Surface-code positions on IBM Quantum: syndrome extraction as LR-QAOA

This notebook runs the **syndrome extraction of a surface-code patch** - every check measured through its
own ancilla, mid-circuit, with feed-forward, round after round - at many **positions on one chip**, and
asks whether the LR-QAOA quality of a patch follows the calibrated error of the qubits it sits on. The
position is the one knob that is free, physical and has nothing to do with the circuit, so it is a direct
test of whether this benchmark ranks parts of a device the way their errors do.

It follows the Helios notebook for codes (`benchmark_codes_quantinuum.ipynb`) with the same structures,
references and analysis. The difference is the chip. IBM's square-lattice devices (Nighthawk, e.g.
`ibm_phoenix`, $12 \times 10$) have fixed couplers, so a patch must be **placed**. The notebook enumerates every
placement, scores each by its calibrated error, and runs a spread of them.

**Safe by default:** nothing is sent to IBM until you set `SUBMIT = True`. Until then the chosen device's
own calibration builds an Aer noise model and a few small placements run locally, flagged
`"simulated": true` and filed as `<device>_sim`. With `BACKEND_NAME = "noisy_simulator"` a synthetic square
lattice with uniform depolarizing noise needs no account at all.

**Requirements**
* `pip install -e ".[ibm]"` from the repository root (or keep the `sys.path` line of the next cell);
* for a real device, an IBM Quantum account saved once with
  `QiskitRuntimeService.save_account(channel="ibm_quantum_platform", token="...", instance="...", name="mcm-primitives")`.

In [1]:
import json
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))          # not needed after `pip install -e .`
DATA, FIGURES = ROOT / "data", ROOT / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)

import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import spearmanr

from qecbench import codes
from qecbench.analysis import load_results
from qecbench.backends import IBMBackend, SimBackend
from qecbench.circuits import build_dynamic
from qecbench.experiment import build_plan, harvest, print_plan, submit
from qecbench.layout import spread_selection, square_lattice_coordinates, surface_code_placements
from qecbench.lrqaoa import ideal_r, random_baseline

## 0. Background

### From a code to a Hamiltonian

A code measured in the $Z$ basis is a set of **checks**. Check $c$ has a support $S_c$ of data qubits, and
the code becomes the Ising Hamiltonian

$$H = \sum_c w_c \prod_{i \in S_c} Z_i .$$

Its term for check $c$ is applied through an ancilla by exactly the syndrome-extraction gadget, with one
phase rotation added: `CX` from every support qubit onto the ancilla copies the parity, `RZ(2 w γ)`
imprints $e^{-i\gamma w_c Z\cdots Z}$, `H` and a **mid-circuit measurement** disentangle it, and when the
outcome is 1 **feed-forward** applies `Z` to the support. So one LR-QAOA layer is one round of syndrome
extraction, and depth $p$ is $p$ rounds. How much of the noiseless answer survives tells how a device
copes with that code's pattern of checks (their weights, overlaps and parallelism) before any
encoding or decoding. The **direct** reference applies the same term with no ancilla, as a `CX` ladder
onto the last support qubit, `RZ(2 w γ)` and the ladder undone: $2(|S_c| - 1)$ `CX`.

Nothing below depends on which code it is. A structure is only `n_data` and its weighted checks
(`qecbench.codes`), so a surface code, a colour code, a qLDPC code or any Hamiltonian you load runs
through the same scheduler, programs, backends and analysis:

| structure | built by | data qubits | checks |
|---|---|---|---|
| surface code | `codes.surface_code(d)` | $d^2$ | $(d-1)^2$ of weight 4, $2(d-1)$ of weight 2 |
| colour code (6.6.6) | `codes.color_code(d)` | $(3d^2+1)/4$ | weight 6 in the bulk, weight 4 on the boundary |
| qLDPC (bivariate bicycle) | `codes.bivariate_bicycle("BB18")`, also `BB24`, `BB30`, `BB48`, `GB16`, `GB26` | 18, 24, ... | independent rows of $H_X$ and $H_Z$, duplicates merged into weight 2 |
| anything else | `codes.from_hamiltonian(name, {(0, 1, 2): 1.0, ...})`, `codes.from_checks`, `codes.load(path)` | | |

`codes.load` reads a structure saved with `CodeStructure.save`, the `hamiltonian` block of an earlier
result file, or a text file with a `#n_qubits:<n>` header and one support per line (`[0, 6, 9, 11, 16]`,
optionally followed by a weight).

### What is measured

* **Approximation ratio** $r = (E_{\max} - \langle E\rangle)/(E_{\max} - E_{\min})$ with its shot-noise error.
  $E_{\min}$ is found exactly (integer programming): for the codes above every check can read $-1$ at once,
  so $E_{\min} = -\sum_c w_c$ and $E_{\max} = +\sum_c w_c$.
* **Random guessing** gives $r_{\rm rand} = 1/2$ exactly, with standard deviation
  $\sqrt{\sum_c w_c^2 / S}\,/\,(E_{\max} - E_{\min})$ at $S$ shots. $n_\sigma = (r - r_{\rm rand})/\sigma$ says
  whether a run carries any signal at all, with no simulation needed.
* **Noiseless reference** $r_{\rm ideal}(p)$, exact up to 25 data qubits (a statevector; above 20 qubits it
  takes seconds to minutes and is kept in `data/references/ideal_energies.json`). Above 25 there is no exact
  reference: a stored estimate is used if there is one, otherwise $r_{\rm ideal}$ and $r_{\rm ovl}$ are NaN
  and $n_\sigma$ is the measure.
* **Normalised quality** $r_{\rm ovl} = (r - r_{\rm rand})/(r_{\rm ideal} - r_{\rm rand})$: 1 is noiseless,
  0 is random.

### Schedules: which checks run together

Two checks that share a data qubit cannot be measured at the same time. `codes.schedule` colours the graph
of those conflicts (DSATUR) and gives **batches** of checks with disjoint supports, which run in parallel:
4 per round for the surface code, 3 for the colour code, 7 for BB18. Fewer batches means fewer
**idle qubit-steps** (data qubits waiting while a batch they are not part of runs), the count reported per
round below.

### Placing a surface-code patch on a square lattice

The surface-code structure has a $d \times d$ grid of data qubits. On a square-lattice chip, data qubit $(i, j)$
sits at lattice site $(i + j + r_0,\; i - j + c_0)$: data qubits become diagonal neighbours, and the data qubits
of every check share a common lattice neighbour, which becomes that check's **ancilla**. Weight-4 checks
take theirs first, each the free common neighbour nearest the patch centre. A placement exists for every
anchor $(r_0, c_0)$ where all these sites are qubits and every (data, ancilla) pair is a coupler. On a
$12 \times 10$ lattice that gives 48 placements for $d = 3$ (17 qubits) and 8 for $d = 5$ (49 qubits).

The circuit is IBM's dynamic-circuit form of the gadget: `H` on the ancillas, the `CZ` of each check in rounds,
`RX(2 w γ)`, the measurement, and a conditional block applying `Z` to the support and `X` to return the
ancilla to $|0\rangle$. Circuits are transpiled with the physical qubits fixed, and a plan that would need any
routing gate is refused. Only the MCM kind runs: in this embedding data qubits are never coupled to each
other, so the direct circuit would need routing.

### Scoring a position

`IBMBackend.error_budget` adds, per layer, the `CZ` error of every coupler the patch uses, the mid-circuit
readout error (`measure_2`) of every ancilla and two `sx` errors per qubit, from the calibration snapshot
taken next to the run. Positions touching a coupler above `MAX_COUPLER_ERROR`, or one that is not
calibrated, are left out. The scan then takes `N_POSITIONS` placements **spread log-evenly over the ranking**
rather than the best ones, because a correlation needs contrast.

## 1. Configuration - the only cell you normally edit

* `BACKEND_NAME` - a square-lattice IBM device, or `"noisy_simulator"` for a synthetic $11 \times 11$ lattice.
* `DISTANCES`, `N_POSITIONS` - patch sizes and how many placements of each to run.
* `MAX_COUPLER_ERROR`, `MAX_READOUT_ERROR` - placements touching anything worse are skipped.
* `DEPTHS`, `SHOTS`, `DELTA` - the LR-QAOA sweep. The defaults follow the earlier `ibm_phoenix` position scan.
* `LOCAL` - what runs when `SUBMIT = False`. Aer simulates dynamic circuits shot by shot on all of a
  patch's qubits, so keep it to $d = 3$ and a few positions (about 15 s per circuit at 200 shots).
* `MAX_CIRCUITS_PER_JOB` - jobs with too many feed-forward operations fail on IBM (error 6073).

In [ ]:
BACKEND_NAME      = "ibm_phoenix"       # a square-lattice device, or "noisy_simulator"
ACCOUNT           = "mcm-primitives"    # saved IBM account; None for the default one
DISTANCES         = [3, 5]
N_POSITIONS       = {3: 12, 5: 8}       # d = 3: a spread of the placements; d = 5: all 8
MAX_COUPLER_ERROR = 0.30
MAX_READOUT_ERROR = 0.30
DEPTHS            = [1, 2, 3, 5, 7, 10]
SHOTS             = 1000
DELTA             = 0.5
LOCAL             = dict(distances=[3], positions=3, depths=[1, 2, 3], shots=200)
OPTIMIZATION_LEVEL   = 3
DYNAMICAL_DECOUPLING = False
MAX_CIRCUITS_PER_JOB = 8
SUBMIT            = False               # True actually submits the jobs and consumes QPU time

## 2. Backend and calibration

Connects to the device, checks that its couplers form a square lattice, and saves a timestamped calibration
snapshot to `data/calibration/`. With `SUBMIT = False` that same calibration builds the local noise model.

In [ ]:
if BACKEND_NAME == "noisy_simulator":
    backend, cal = SimBackend(topology="grid", qubits=121, seed=1), None
else:
    backend = IBMBackend(BACKEND_NAME, account=ACCOUNT, optimization_level=OPTIMIZATION_LEVEL,
                         dynamical_decoupling=DYNAMICAL_DECOUPLING, max_circuits_per_job=MAX_CIRCUITS_PER_JOB,
                         local=not SUBMIT, seed=1)
    print(f"mid-circuit measurement: {backend.supports_mcm()}")
    cal = backend.calibration(save_dir=DATA / "calibration")
G = backend.coupling_graph()
coords = square_lattice_coordinates(G)
assert coords is not None, f"{backend.name}: couplers do not form a square lattice"
print(f"{backend.name}: {G.number_of_nodes()} qubits on a square lattice"
      + (f"  ({backend.noise_description})" if backend.simulated else ""))
distances, depths, shots = ((DISTANCES, DEPTHS, SHOTS) if SUBMIT else
                            (LOCAL["distances"], LOCAL["depths"], LOCAL["shots"]))

## 3. Positions

Every placement of each patch, the ones touching a bad coupler or readout left out, scored by the error
budget, and the scanned spread. The maps show the scanned patches on the chip, their couplers coloured by
the patch's budget (yellow is lowest, dark purple highest). Without a calibration (`noisy_simulator`) there is no budget: the spread
is taken in placement order and each patch gets its own colour.

In [ ]:
positions = {}
for d in distances:
    code = codes.surface_code(d)
    placements = surface_code_placements(G, code)
    if cal is not None:
        flagged = {patch for patch, _ in backend.flag_instances(placements, cal, max_2q_error=MAX_COUPLER_ERROR,
                                                                max_readout_error=MAX_READOUT_ERROR)}
        usable = [patch for patch in placements if patch not in flagged]
        scores = [backend.error_budget(patch, cal) for patch in usable]
    else:
        usable, scores = placements, list(range(len(placements)))
    k = N_POSITIONS.get(d, len(usable)) if SUBMIT else LOCAL["positions"]
    positions[d] = spread_selection(usable, scores, k)
    print(f"d = {d}: {len(placements)} placements, {len(usable)} usable -> scanning {len(positions[d])}")
    if cal is not None and usable:
        print(f"   error budget per layer from {min(scores):.3f} to {max(scores):.3f} "
              f"({max(scores) / min(scores):.1f}x)")
    for patch, score in positions[d]:
        print(f"   {patch}  budget {score:.3f}" if cal is not None else f"   {patch}")

fig, axes = plt.subplots(1, len(distances), figsize=(5.5 * len(distances), 6), squeeze=False)
for ax, d in zip(axes[0], distances):
    for u, v in G.edges:
        ax.plot([coords[u][1], coords[v][1]], [-coords[u][0], -coords[v][0]], color="0.9", lw=1, zorder=0)
    budgets = [score for _, score in positions[d]]
    norm = plt.Normalize(min(budgets), max(budgets) + 1e-12)
    for k, (patch, score) in enumerate(positions[d]):
        colour = plt.cm.viridis_r(norm(score)) if cal is not None else f"C{k}"
        for u, v in patch.couplers:
            ax.plot([coords[u][1], coords[v][1]], [-coords[u][0], -coords[v][0]], color=colour, lw=2.5,
                    alpha=0.6, zorder=1)
    xy = np.array([(coords[q][1], -coords[q][0]) for q in G.nodes])
    ax.scatter(*xy.T, s=12, color="0.6", zorder=2)
    ax.set_aspect("equal")
    ax.axis("off")
    ax.set_title(f"d = {d}: {len(positions[d])} positions")
plt.show()
instances = [patch for d in distances for patch, _ in positions[d]]

## 4. Plan and cost

One circuit per position and depth (`max_per_batch=1`): patches never share a circuit, so the position is
the only thing that changes between them. `build_plan` transpiles every circuit with its qubits fixed and
refuses any routing. The printout gives the QPU-time estimate (the feed-forward time per conditional is an
unverified assumption; IBM's usage report is authoritative).

In [ ]:
plan = build_plan(backend, instances, depths=depths, shots=shots, delta=DELTA, max_per_batch=1)
print_plan(plan, backend)
print("\nnoiseless r and random + 3 sigma at these shots:")
for d in distances:
    patch = positions[d][0][0]
    r_rand, sigma = random_baseline(patch, shots)
    print(f"   d = {d}: " + ", ".join(f"p={p}: {ideal_r(patch, p, DELTA):.3f}" for p in depths)
          + f"   (random + 3 sigma = {r_rand + 3 * sigma:.3f})")

## 5. Inspect, then submit

The logical circuit of the smallest patch at $p = 1$, then the submission. With `SUBMIT = False` the plan is
simulated locally under the device's calibration: same circuits, same bookkeeping, no QPU time. A manifest is
written after every job.

In [ ]:
print(build_dynamic([instances[0]], 1, DELTA).draw(output="text", fold=140))

In [ ]:
manifest_path = submit(plan, backend, manifest_dir=DATA / "manifests")

## 6. Harvest

Needs only the manifest. Results go to `data/results/<backend>/surface_code/mcm/`, one file per run, each
result keeping the placement (data qubits and ancillas) it ran on.

In [ ]:
manifests = sorted((DATA / "manifests" / backend.name).glob("*.json"), key=lambda p: p.name)
assert manifests, "no manifest yet - run the submit cell first"
manifest_path = manifests[-1]
print("harvesting", manifest_path.name)
saved = harvest(manifest_path, backend, data_dir=DATA / "results")

## 7. Does the position show?

Left: $r_{\rm ovl}$ against depth, one line per position, coloured by its error budget (yellow is lowest, dark purple highest), with
the median over positions. Right: $r_{\rm ovl}$ at the deepest common depth against the error budget, with
Spearman's rank correlation. If the benchmark ranks positions the way their calibrated errors do, the
correlation is negative. The budget comes from the newest calibration snapshot of this device taken before
the run, so the cell also works in a later session.

In [ ]:
results = load_results(DATA / "results", backend.name, kind="mcm", manifest_path=manifest_path)
created = json.loads(Path(manifest_path).read_text())["created"]
snapshots = sorted((DATA / "calibration" / backend.name).glob("*_calibration.json"))
before = [p for p in snapshots if json.loads(p.read_text())["fetched_at"] <= created]
run_cal = json.loads(before[-1].read_text()) if before else cal
budget = {patch: backend.error_budget(patch, run_cal) if run_cal else float("nan") for patch in results}

by_d = {}
for patch, by_depth in results.items():
    by_d.setdefault(patch.code.name, []).append((patch, by_depth))
fig, axes = plt.subplots(len(by_d), 2, figsize=(12, 4.3 * len(by_d)), squeeze=False)
for (ax, bx), (name, rows) in zip(axes, sorted(by_d.items())):
    finite = [budget[p] for p, _ in rows if np.isfinite(budget[p])]
    norm = plt.Normalize(min(finite), max(finite) + 1e-12) if finite else None
    ps = sorted(set.intersection(*(set(by) for _, by in rows)))
    for patch, by_depth in rows:
        colour = plt.cm.viridis_r(norm(budget[patch])) if norm else "C0"
        ax.plot(ps, [by_depth[p]["r_ovl"] for p in ps], "o-", color=colour, alpha=0.8, ms=4)
    ax.plot(ps, [np.median([by[p]["r_ovl"] for _, by in rows]) for p in ps], "s--", color="black", lw=2,
            label="median over positions")
    ax.axhline(0, color="0.5", lw=0.8)
    ax.set(xlabel="LR-QAOA layers $p$", ylabel=r"$r_{\rm ovl}$", title=f"{backend.name}, {name}: {len(rows)} positions",
           xticks=ps)
    ax.legend(frameon=False)
    deepest = ps[-1]
    x = np.array([budget[p] for p, _ in rows])
    y = np.array([by[deepest]["r_ovl"] for _, by in rows])
    err = np.array([by[deepest]["r_err"] / (by[deepest]["r_ideal"] - by[deepest]["r_rand"]) for _, by in rows])
    if not np.isfinite(x).all():
        bx.text(0.5, 0.5, "no calibration, no error budget", ha="center", va="center", transform=bx.transAxes)
        bx.set_axis_off()
        continue
    bx.errorbar(x, y, yerr=err, fmt="o", mec="black", capsize=3)
    if len(rows) >= 3:
        rho, pvalue = spearmanr(x, y)
        bx.set_title(f"p = {deepest}: Spearman rho = {rho:+.2f} (P = {pvalue:.3f}, n = {len(rows)})")
    bx.set(xlabel="error budget per layer", ylabel=rf"$r_{{\rm ovl}}$ at p = {deepest}")
fig.tight_layout()
fig.savefig(FIGURES / f"{backend.name}_surface_code_positions.pdf", bbox_inches="tight")
plt.show()